# 🗳️ Multi-Method Hinglish News Clustering Playbook
---
This notebook contains all the methodologies we developed to organize **9,200+ Hinglish video titles** into actionable news topics.

### **The Journey**
1. **Baseline**: Pure semantic discovery.
2. **Expert Engine**: High-granularity sub-topic detection.
3. **Entity Anchored**: Person-first grouping for Search API stability.
4. **Zero-Noise**: Force-assignment of outliers.

In [ ]:
# Install Dependencies
!pip install bertopic sentence-transformers hdbscan umap-learn pandas tqdm scikit-learn requests

import pandas as pd
import numpy as np
import re, html, json
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import umap, hdbscan

print("✅ Setup Complete")

## 🛠️ The Master Hinglish Cleaner
This function handles mixed-script noise and protects Hindi tokens.

In [ ]:
def normalize_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # URLs
    text = re.sub(r'#(\w+)', r' \1 ', text)           # Hashtags
    text = re.sub(r'[^\w\s\u0900-\u097F]', ' ', text) # Preserve Devanagari
    tokens = [t for t in text.lower().split() if not (t.isascii() and len(t) <= 2)]
    return " ".join(tokens).strip()

# Sample Usage
sample = "Breaking News: ईरान-इजरायल वार #IranWar live updates at2"
print(f"Raw: {sample}")
print(f"Clean: {normalize_text(sample)}")

## Method 1: Semantic Exploration
**Intuition**: Let the model find natural clusters without bias.

In [ ]:
def method_1_run(docs, embeddings):
    model = BERTopic(
        embedding_model="sentence-transformers/LaBSE",
        min_cluster_size=15,
        verbose=True
    )
    topics, probs = model.fit_transform(docs, embeddings)
    return model, topics

## Method 3: Entity-Anchored (Production Recommended)
**Intuition**: Anchor topics to Top Persons (Modi, Trump) and reduce noise to 0%.

In [ ]:
seeds = [["modi", "pm", "bjp"], ["trump", "iran", "war"], ["rahul", "gandhi", "congress"]]

def method_3_run(docs, embeddings):
    model = BERTopic(
        seed_topic_list=seeds,
        min_cluster_size=15,
        verbose=True
    )
    topics, _ = model.fit_transform(docs, embeddings)
    
    # Force assign outliers
    new_topics = model.reduce_outliers(docs, topics, strategy="embeddings")
    model.update_topics(docs, topics=new_topics)
    return model, new_topics